# PIE paper-weights reproduction on the full test split (Colab)

Runs the published Keras `.h5` checkpoints through our PyTorch port on **set03 (the paper's test split)** and reports intent / trajectory / speed metrics against the paper's published numbers.

## Paper reference numbers (ICCV 2019, Table 2 / 4 / 5 on set03)

| Model | Metric | Paper value | Committed ckpt |
|---|---|---|---|
| Intent (`PIEint`, imgcontext + loc) | test accuracy | **0.79** | `intention/context_loc_pretrained/model.h5` |
| Intent (`PIEint`, imgcontext + loc) | test F1 | **0.87** | same |
| Trajectory (`PIEtraj`, loc + int + speed) | MSE at 1.5s (pixels) | **473** | `trajectory/loc_intent_speed_pretrained/model.h5` |
| Trajectory (`PIEtraj`, loc + int + speed) | CMSE at 1.5s | **435** | same |
| Trajectory (`PIEtraj`, loc + int + speed) | CFMSE at 1.5s | **1741** | same |
| Speed (`PIEspeed`) | MSE at 1.5s (km/h) | **2.65** | `speed/speed_pretrained/model.h5` |

Filename values like `0.81.pkl` and `473.14.pkl` are the best **val** metrics reached during paper training, which differ slightly from the final **test** metrics reported in the paper.

## What the notebook does for you

Everything that can be done in code is done in code (Drive mount, repo clone, pip install, annotation download, video download, frame extraction, weight conversion, evaluation). Preconditions that cannot be fixed programmatically (no GPU, not enough Drive space, repo missing committed files) raise a `RuntimeError` immediately rather than failing silently downstream.

## Resource budget

A full set03 pass needs ~35 GB Drive free (~25 GB MP4s + ~15-25 GB extracted frames + ~8 GB VGG16 feature cache). The precondition cell below hard-asserts this.

Wall-clock on a T4:
- set03 videos download: ~5-15 min depending on bandwidth
- set03 frame extraction: ~15-30 min
- Intent VGG feature caching: ~30-60 min (one-shot)
- Evals after caches are warm: minutes each

First run: ~2-3 hours end-to-end. Subsequent runs reuse caches and finish in ~10 minutes.


## 1. Runtime + Drive

The runtime-check cell `raise`s (not warns) if there's no GPU — CPU eval of the intent model is impractical.

In [ ]:
import subprocess
# Hard check: GPU must be present.
try:
    out = subprocess.check_output(['nvidia-smi'], text=True, timeout=10)
    print(out.split('\n')[0])  # header line
except (FileNotFoundError, subprocess.CalledProcessError) as e:
    raise RuntimeError(
        'No GPU detected. Runtime -> Change runtime type -> T4 GPU, '
        'then re-run this cell.'
    ) from e

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.environ['PIE_PATH'] = '/content/drive/MyDrive/Portofolios/Protofolio_2026/pedestrian_estimation_pie/pie_data'
print('PIE_PATH =', os.environ['PIE_PATH'])

## 2. Clone + install

In [ ]:
%cd /content
![ -d PIE_Pedestrian_Estimation ] || git clone -b claude/tensorflow-to-pytorch-conversion-0SlwI https://github.com/venetisgr/PIE_Pedestrian_Estimation.git
%cd /content/PIE_Pedestrian_Estimation
!git checkout claude/tensorflow-to-pytorch-conversion-0SlwI
!git pull
!git log --oneline -3

In [ ]:
!pip install -q -r requirements.txt

## 3. Preconditions

Every cell below is a hard check. Any failure is a **red-bordered `raise`**, not a warning. Where a precondition can be fixed automatically (download annotations / videos, extract frames), the cell runs the fix **and then re-asserts** the result.

In [ ]:
# 3a. PIE_PATH exists (auto-created if missing) + Drive space sufficient
# + paper checkpoints committed. Anything we can fix is done in code;
# only things that must be user-side (Colab Drive quota, git state) raise.
import shutil
from pathlib import Path

PIE_PATH = Path(os.environ['PIE_PATH'])
PIE_PATH.mkdir(parents=True, exist_ok=True)

MIN_FREE_GB = 35
free_gb = shutil.disk_usage(str(PIE_PATH)).free / 2**30
print(f'Drive free space at PIE_PATH: {free_gb:.1f} GB')
if free_gb < MIN_FREE_GB:
    raise RuntimeError(
        f'Only {free_gb:.1f} GB free at {PIE_PATH}. A full set03 pass needs '
        f'at least {MIN_FREE_GB} GB (25 GB MP4s + 15-25 GB frames + 8 GB VGG cache).'
    )

H5 = {
    'intent':     Path('data/pie/intention/context_loc_pretrained/model.h5'),
    'trajectory': Path('data/pie/trajectory/loc_intent_speed_pretrained/model.h5'),
    'speed':      Path('data/pie/speed/speed_pretrained/model.h5'),
}
missing_h5 = [str(p) for p in H5.values() if not p.is_file()]
if missing_h5:
    raise FileNotFoundError(
        f'Paper .h5 checkpoints missing from the repo: {missing_h5}. '
        'These are committed files; re-clone the repo or check git lfs.'
    )
print('3a OK: PIE_PATH + Drive space + .h5 checkpoints')


In [ ]:
# 3b. Annotations present for set03. Auto-fetch, then re-assert.
ANNOT_DIRS = ['annotations', 'annotations_attributes', 'annotations_vehicle']
def _set03_annotations_ok() -> list[str]:
    return [d for d in ANNOT_DIRS if not (PIE_PATH / d / 'set03').is_dir()]

missing = _set03_annotations_ok()
if missing:
    print(f'annotation dirs missing for set03: {missing}. Fetching...')
    !python -m pie_pytorch.data.downloader annotations --dest "$PIE_PATH" --overwrite
    still_missing = _set03_annotations_ok()
    if still_missing:
        raise FileNotFoundError(
            f'Auto-fetch ran but {still_missing} still missing under {PIE_PATH}. '
            'Check the downloader output above for errors.'
        )
print('3b OK: all three annotation dirs contain set03/')

In [ ]:
# 3c. set03 videos (19 MP4s). Auto-fetch, then re-assert.
import glob
EXPECTED_SET03_VIDEOS = 19
set03_dir = PIE_PATH / 'PIE_clips' / 'set03'
def _n_videos() -> int:
    return len(sorted(glob.glob(str(set03_dir / 'video_*.mp4'))))

n = _n_videos()
print(f'set03 videos on disk: {n}/{EXPECTED_SET03_VIDEOS}')
if n < EXPECTED_SET03_VIDEOS:
    print('Downloading missing set03 videos (resumable; ~25 GB total)...')
    !python -m pie_pytorch.data.downloader videos --dest "$PIE_PATH" --sets set03 --workers 4
    n = _n_videos()
    if n != EXPECTED_SET03_VIDEOS:
        raise RuntimeError(
            f'After download, set03 still has {n}/{EXPECTED_SET03_VIDEOS} videos. '
            'Re-run this cell; the downloader is resumable.'
        )
print(f'3c OK: all {EXPECTED_SET03_VIDEOS} set03 videos present')

In [ ]:
# 3d. Frames extracted for every annotated video in set03.
# PIE.get_annotated_frame_numbers returns [count, frame_1, frame_2, ...]
# per video; the first element is the number of annotated frames, NOT
# a frame index. The extraction loop in pie_data uses `frames[1:]` for
# that reason; our check must too.
from pie_pytorch.data.pie_data import PIE
imdb = PIE(data_path=str(PIE_PATH))

def _missing_frame_videos() -> list[str]:
    annotated = imdb.get_annotated_frame_numbers('set03')
    missing = []
    for vid, raw in annotated.items():
        frame_ids = raw[1:]  # drop the count prefix
        if not frame_ids:
            continue
        vdir = PIE_PATH / 'images' / 'set03' / vid
        if not vdir.is_dir():
            missing.append(vid)
            continue
        sample = sorted(frame_ids)[:3]
        if not all((vdir / f'{fi:05d}.png').is_file() for fi in sample):
            missing.append(vid)
    return missing

miss = _missing_frame_videos()
if miss:
    print(f'{len(miss)} set03 videos need frame extraction; running one-shot (~15-30 min)...')
    imdb.extract_and_save_images(extract_frame_type='annotated')
    miss = _missing_frame_videos()
    if miss:
        raise RuntimeError(
            f'Extraction ran but {len(miss)} videos still missing frames: {miss[:5]}... '
            'Check MP4 integrity and re-run.'
        )
print('3d OK: every set03 annotated video has matching PNGs')


In [ ]:
# 3e. Final green-light summary.
ann = imdb.get_annotated_frame_numbers('set03')
n_vids = len(ann)
n_frames = sum(len(v) for v in ann.values())
if n_vids != EXPECTED_SET03_VIDEOS:
    raise AssertionError(
        f'set03 annotation count wrong: {n_vids} videos, expected {EXPECTED_SET03_VIDEOS}'
    )
print(f'3e OK: set03 annotated -> {n_vids} videos, {n_frames} total frames')
print('All preconditions satisfied. Ready to evaluate.')

## 4. Convert the three `.h5` checkpoints (one-shot)

In [ ]:
!python -m pie_pytorch.cli.convert \
    --task intent \
    --h5 data/pie/intention/context_loc_pretrained/model.h5 \
    --out data/pie/intention/context_loc_pretrained/model.safetensors
!python -m pie_pytorch.cli.convert \
    --task trajectory \
    --h5 data/pie/trajectory/loc_intent_speed_pretrained/model.h5 \
    --out data/pie/trajectory/loc_intent_speed_pretrained/model.safetensors
!python -m pie_pytorch.cli.convert \
    --task speed \
    --h5 data/pie/speed/speed_pretrained/model.h5 \
    --out data/pie/speed/speed_pretrained/model.safetensors

## 5. Evaluate paper weights on **set03** (paper's test split)

### 5a. Intent (paper: test accuracy 0.79, F1 0.87 — Table 2, `imgcontext + loc`)

First run caches VGG16 features per window — slow (~30-60 min). Subsequent runs hit the cache.


In [ ]:
!python -m pie_pytorch.cli.eval \
    --config pie_pytorch/configs/intent_colab.yaml \
    --keras-h5 data/pie/intention/context_loc_pretrained/model.h5 \
    --split test

### 5b. Trajectory (paper: MSE 473, CMSE 435, CFMSE 1741 at 1.5s — Table 5, `loc + int + speed`)

Note: Table 5's 473 is reported with **ground-truth** intent and speed fed as decoder inputs. With the *predicted* int+speed from PIEint + PIEspeed the paper reports MSE 559 (Table 5, `loc + PIEint + PIEspeed`). Our eval here uses whatever is in the annotations (intention_prob is a per-pedestrian GT, obd_speed is sensor GT), so expect the number to track 473 more closely than 559.


In [ ]:
!python -m pie_pytorch.cli.eval \
    --config pie_pytorch/configs/trajectory_colab.yaml \
    --keras-h5 data/pie/trajectory/loc_intent_speed_pretrained/model.h5 \
    --split test

### 5c. Speed (paper: MSE 2.65 at 1.5s, km/h — Table 4)

The paper trained speed with a 1-dim zero-filled decoder input; our `speed_colab.yaml` uses `dec_feature_size: 0` for scratch-training. `pie-eval --keras-h5` autosizes the model from the `.h5` shapes (which encode `dec_feature_size=1`), so the eval is correct regardless.


In [ ]:
!python -m pie_pytorch.cli.eval \
    --config pie_pytorch/configs/speed_colab.yaml \
    --keras-h5 data/pie/speed/speed_pretrained/model.h5 \
    --split test

## Done

Paste the three metric blocks back to the thread. Paper targets (set03 test split):

| Model | Metric | Paper | Acceptable range |
|---|---|---|---|
| Intent | acc | 0.79 | 0.75-0.82 |
| Intent | F1 | 0.87 | 0.83-0.90 |
| Trajectory | MSE at 1.5s | 473 | 420-550 |
| Trajectory | CMSE at 1.5s | 435 | 380-490 |
| Speed | MSE at 1.5s | 2.65 | 2.3-3.1 |

Outside the acceptable range → paste full eval output + `!du -sh $PIE_PATH/*`.
